# 實踐項目 1：數據分析基礎

## 🎯 項目目標

通過分析真實數據集，綜合運用張量操作、Pandas 數據處理和統計分析技能。

## 📚 涵蓋知識點

- PyTorch 張量基礎操作
- Pandas 數據清洗與轉換
- 統計分析（均值、方差、分佈）
- 數據可視化
- 數據標準化與歸一化

## 🔧 環境準備

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 設置繪圖樣式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ 環境準備完成！")

## 📊 任務 1：數據加載與探索

我們將使用經典的 Wine 數據集（葡萄酒數據集）。

In [ ]:
# 加載數據集
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print("數據集基本信息：")
print(f"樣本數量: {len(df)}")
print(f"特徵數量: {len(wine.feature_names)}")
print(f"類別數量: {len(wine.target_names)}")
print(f"\n類別名稱: {wine.target_names}")

# 顯示前幾行
print("\n數據前5行：")
df.head()

### 🎯 練習 1.1：數據基本統計

**任務**：計算每個特徵的基本統計量（均值、標準差、最小值、最大值）

In [ ]:
# TODO: 使用 Pandas 計算統計量
stats = df.describe()
print("基本統計量：")
stats

In [ ]:
# TODO: 使用 PyTorch 張量計算同樣的統計量
data_tensor = torch.tensor(df.drop('target', axis=1).values, dtype=torch.float32)

mean = data_tensor.mean(dim=0)
std = data_tensor.std(dim=0)
min_val = data_tensor.min(dim=0)[0]
max_val = data_tensor.max(dim=0)[0]

print("\n使用 PyTorch 計算的統計量：")
print(f"均值: {mean[:3]}...")  # 只顯示前3個
print(f"標準差: {std[:3]}...")
print(f"最小值: {min_val[:3]}...")
print(f"最大值: {max_val[:3]}...")

## 📊 任務 2：數據可視化

可視化是理解數據的重要手段。

In [ ]:
# 特徵分佈可視化
fig, axes = plt.subplots(3, 5, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(wine.feature_names):
    axes[idx].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[idx].set_title(col, fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

# 移除多餘的子圖
for idx in range(len(wine.feature_names), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('wine_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 特徵分佈圖已保存")

### 🎯 練習 2.1：相關性分析

**任務**：計算並可視化特徵之間的相關性

In [ ]:
# TODO: 計算相關係數矩陣
correlation_matrix = df.drop('target', axis=1).corr()

# 可視化相關性矩陣
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 相關性矩陣圖已保存")

## 🔧 任務 3：數據預處理

### 3.1 數據標準化（Z-score Normalization）

公式：$z = \frac{x - \mu}{\sigma}$

In [ ]:
def standardize(data):
    """
    標準化數據（Z-score）
    
    參數:
        data: torch.Tensor, 形狀為 (n_samples, n_features)
    
    返回:
        標準化後的數據
    """
    # TODO: 實現標準化
    mean = data.mean(dim=0, keepdim=True)
    std = data.std(dim=0, keepdim=True)
    return (data - mean) / (std + 1e-8)  # 加小數避免除以零

# 測試標準化
data_tensor = torch.tensor(df.drop('target', axis=1).values, dtype=torch.float32)
standardized_data = standardize(data_tensor)

print("標準化前：")
print(f"均值: {data_tensor.mean(dim=0)[:3]}")
print(f"標準差: {data_tensor.std(dim=0)[:3]}")

print("\n標準化後：")
print(f"均值: {standardized_data.mean(dim=0)[:3]}")
print(f"標準差: {standardized_data.std(dim=0)[:3]}")

### 🎯 練習 3.1：實現 Min-Max 歸一化

**任務**：實現 Min-Max 歸一化，將數據縮放到 [0, 1] 範圍

公式：$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$

In [ ]:
def min_max_normalize(data):
    """
    Min-Max 歸一化
    
    參數:
        data: torch.Tensor
    
    返回:
        歸一化後的數據（範圍 [0, 1]）
    """
    # TODO: 實現 Min-Max 歸一化
    min_val = data.min(dim=0, keepdim=True)[0]
    max_val = data.max(dim=0, keepdim=True)[0]
    return (data - min_val) / (max_val - min_val + 1e-8)

# 測試
normalized_data = min_max_normalize(data_tensor)

print("歸一化後的數據範圍：")
print(f"最小值: {normalized_data.min(dim=0)[0][:3]}")
print(f"最大值: {normalized_data.max(dim=0)[0][:3]}")

## 📊 任務 4：統計分析

### 4.1 按類別分組分析

In [ ]:
# 按類別分組統計
grouped_stats = df.groupby('target').agg(['mean', 'std'])
print("按類別的統計量：")
print(grouped_stats.iloc[:, :6])  # 只顯示部分列

# 可視化不同類別的特徵分佈
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# 選擇6個最具代表性的特徵
features_to_plot = wine.feature_names[:6]

for idx, feature in enumerate(features_to_plot):
    for target in range(3):
        subset = df[df['target'] == target][feature]
        axes[idx].hist(subset, bins=20, alpha=0.5, label=wine.target_names[target])
    
    axes[idx].set_title(feature, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('class_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 類別分佈圖已保存")

### 🎯 練習 4.1：協方差矩陣計算

**任務**：使用 PyTorch 計算協方差矩陣

協方差公式：$\text{Cov}(X, Y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$

In [ ]:
def compute_covariance_matrix(data):
    """
    計算協方差矩陣
    
    參數:
        data: torch.Tensor, 形狀為 (n_samples, n_features)
    
    返回:
        協方差矩陣, 形狀為 (n_features, n_features)
    """
    # TODO: 實現協方差矩陣計算
    # 1. 中心化數據（減去均值）
    centered = data - data.mean(dim=0, keepdim=True)
    
    # 2. 計算協方差矩陣
    n = data.shape[0]
    cov_matrix = (centered.T @ centered) / (n - 1)
    
    return cov_matrix

# 測試
cov_matrix = compute_covariance_matrix(data_tensor)
print(f"協方差矩陣形狀: {cov_matrix.shape}")
print(f"\n協方差矩陣（前3×3）:\n{cov_matrix[:3, :3]}")

## 🎓 任務 5：綜合應用

### 5.1 實現簡單的數據異常檢測

使用 Z-score 方法檢測異常值（|Z| > 3）

In [ ]:
def detect_outliers_zscore(data, threshold=3.0):
    """
    使用 Z-score 方法檢測異常值
    
    參數:
        data: torch.Tensor
        threshold: float, Z-score 閾值
    
    返回:
        異常值的布爾掩碼
    """
    # TODO: 實現異常檢測
    mean = data.mean(dim=0, keepdim=True)
    std = data.std(dim=0, keepdim=True)
    z_scores = torch.abs((data - mean) / (std + 1e-8))
    
    # 任何特徵的 Z-score > threshold 即為異常
    outliers = (z_scores > threshold).any(dim=1)
    
    return outliers

# 檢測異常值
outliers = detect_outliers_zscore(data_tensor)
n_outliers = outliers.sum().item()

print(f"檢測到 {n_outliers} 個異常樣本（總共 {len(data_tensor)} 個樣本）")
print(f"異常比例: {n_outliers / len(data_tensor) * 100:.2f}%")

# 顯示異常樣本的索引
if n_outliers > 0:
    outlier_indices = torch.where(outliers)[0]
    print(f"\n異常樣本索引: {outlier_indices[:10].tolist()}...")  # 只顯示前10個

## 📝 項目總結

### 你學到了什麼？

✅ PyTorch 張量的基礎操作和統計計算  
✅ Pandas 數據處理和分組分析  
✅ 數據可視化技巧  
✅ 數據預處理方法（標準化、歸一化）  
✅ 協方差矩陣計算  
✅ 異常值檢測  

### 🎯 進階挑戰

1. **特徵工程**：創建新的組合特徵
2. **降維分析**：實現 PCA（主成分分析）
3. **統計檢驗**：使用 t-test 比較不同類別的特徵
4. **交互式可視化**：使用 Plotly 創建交互式圖表

### 📚 推薦閱讀

- [PyTorch 官方文檔 - Tensor Operations](https://pytorch.org/docs/stable/torch.html)
- [Pandas 數據分析教程](https://pandas.pydata.org/docs/user_guide/index.html)
- [Statistics for Machine Learning](https://machinelearningmastery.com/statistics_for_machine_learning/)

---

**恭喜完成項目 1！🎉**

接下來嘗試 [項目 2：神經網絡數學基礎](02_neural_network_math.ipynb)